# Comparing coords2sans with SasView and sasmodels

coords2sans should give the same curves as the programs people use
today. This notebook runs four checks:

1. **The exact average against SasView:** the Debye sum over all
   atom pairs, which coords2sans computes by default, against
   SasView's own sum, on the same micelle.
2. **The averaged amplitude against SasView:** the same micelle
   through SasView's way (a PDB file, `PDBReader`, then `GenSAS`
   with `is_avg = True`), as the polyJD script does it, and through
   coords2sans with `average_amplitude = True`. Then how far that
   is from the exact average for this rod micelle.
3. **Polydispersity against sasmodels:** a sphere of evenly spread
   atoms with a 10 % spread of sizes, against the `sphere` model of
   sasmodels (the model library of SasView) with the same spread.
4. **The polyJD recipe:** why coords2sans does not give the polyJD
   script's polydisperse curve.

It needs `sasmodels` (in the dev environment) and three files of
SasView's calculator: `sas_gen.py`, `geni.py` and
`sasview_sans_debye.py`. They are downloaded once from SasView's
GitHub at a fixed commit, so the result does not change when
SasView changes. SasView itself, with its GUI, is not needed.

In [ ]:
import sys
import tempfile
import urllib.request
from pathlib import Path
from statistics import NormalDist

import numpy as np
from sasmodels.core import load_model
from sasmodels.data import empty_data1D
from sasmodels.direct_model import DirectModel

from coords2sans import SansCalculator
from coords2sans import Structure

In [ ]:
# the SasView commit of 2026-01-15, the last change to sas_gen.py
SASVIEW_COMMIT = '02363ba1639ddaf4005099e24472947858f4c99d'
SASVIEW_URL = f'https://raw.githubusercontent.com/SasView/sasview/{SASVIEW_COMMIT}/src/'
sasview_src = Path(tempfile.gettempdir()) / f'sasview-{SASVIEW_COMMIT[:7]}'
calculator_dir = sasview_src / 'sas' / 'sascalc' / 'calculator'
calculator_dir.mkdir(parents=True, exist_ok=True)
(calculator_dir / 'ausaxs').mkdir(exist_ok=True)
for name in ('sas_gen.py', 'geni.py', 'ausaxs/sasview_sans_debye.py'):
    path = calculator_dir / name
    if not path.exists():
        part = path.with_suffix('.part')
        urllib.request.urlretrieve(SASVIEW_URL + f'sas/sascalc/calculator/{name}', part)  # noqa: S310
        part.rename(path)
# empty package files: SasView's own sas/__init__.py loads its settings
for package in (sasview_src / 'sas', calculator_dir.parent, calculator_dir, path.parent):
    (package / '__init__.py').touch()
sys.path.insert(0, str(sasview_src))

In [ ]:
from sas.sascalc.calculator import sas_gen
from sas.sascalc.calculator.ausaxs.sasview_sans_debye import sasview_sans_debye

## The micelle

The same CTAB rod micelle as in the `usage` notebook, after the last
equilibration stage, with its elements from the CTA masses.

In [ ]:
N, C, H = 14.007, 12.011, 1.008
cta_masses = [N] + [C, H, H, H] * 3 + [C, H, H] * 15 + [C, H, H, H]
structure = Structure.read(
    Path('data/CTA178-eq5.gro'), residues=['CTA'], masses={'CTA': cta_masses}
)
structure.element_sources

## 1. The exact average against SasView

Micelles in a sample point in all directions. The Debye sum averages
the intensity over all of them exactly: Σ w_j w_k sin(q r_jk) /
(q r_jk) over all atom pairs, with w = (SLD − solvent SLD) · volume.
`GenSAS` with `is_avg = False` computes it with the AUSAXS library,
or, when that is not installed, with `sasview_sans_debye`: every pair
at every q. coords2sans first counts the pairs by distance, in steps
of `SansCalculator.BIN_WIDTH` (0.01 Å), and then sums over the
steps.

SasView's sum takes about 45 s per contrast here, so this check uses
32 q values.

In [ ]:
ax = None
for contrast, solvent_sld in {'H2O': -0.56e-6, 'D2O': 6.36e-6}.items():
    curve = SansCalculator(solvent_sld, q_points=32).calculate(
        structure, f'coords2sans, {contrast}'
    )
    amplitudes = (structure.sld() - solvent_sld) * structure.volume
    # GenSAS's 1D intensity: the sum, times 1e8 / Σ V (in geni.Iq)
    reference = (
        sasview_sans_debye(curve.q, structure.positions.T, amplitudes)
        * 1e8
        / structure.volume.sum()
    )
    difference = np.abs(curve.intensity / reference - 1).max()
    print(f'{contrast}: largest relative difference {difference:.1e}')
    ax = curve.plot(ax)
    ax.loglog(curve.q, reference, 'k:', label=f'SasView exact sum, {contrast}')
ax.legend();

## 2. The averaged amplitude against SasView

The polyJD workflow gives SasView a PDB file. `PDBReader` finds each
atom's element from its name and gives it SasView's SLD and volume;
`GenSAS` with `is_avg = True` averages the amplitude over the
directions and then squares it. Here coords2sans writes its micelle
as such a file: `PDBReader` takes the element from columns 13 to 16,
so a one-letter element starts in column 14.

In [ ]:
pdb_path = Path(tempfile.mkdtemp()) / 'CTA178-eq5.pdb'
atom_lines = [
    f'ATOM  {number:5d}  {element:<3} CTA A   1    {x:8.3f}{y:8.3f}{z:8.3f}  1.00  0.00'
    for number, (element, (x, y, z)) in enumerate(
        zip(structure.elements, structure.positions, strict=True), 1
    )
]
pdb_path.write_text('\n'.join([*atom_lines, 'END', '']))

In [ ]:
ax = None
for contrast, solvent_sld in {'H2O': -0.56e-6, 'D2O': 6.36e-6}.items():
    calculator = SansCalculator(solvent_sld, average_amplitude=True)
    curve = calculator.calculate(structure, f'coords2sans, {contrast}')
    sas_model = sas_gen.GenSAS()
    sas_model.params['solvent_SLD'] = solvent_sld
    sas_model.set_sld_data(sas_gen.PDBReader().read(str(pdb_path)))
    sas_model.is_avg = True
    reference = sas_model.run([curve.q, []])
    difference = np.abs(curve.intensity / reference - 1).max()
    print(f'{contrast}: largest relative difference {difference:.1e}')
    ax = curve.plot(ax)
    ax.loglog(curve.q, reference, 'k:', label=f'SasView GenSAS, {contrast}')
ax.legend();

The averaged amplitude is exact only for a round object. This
micelle is a short rod, about 99 × 64 × 56 Å. Up to q ≈ 0.04 Å⁻¹
both averages agree; above that the averaged amplitude falls too
low and shows a deep minimum the micelle does not have. The polyJD
script's curves have the same error.

In [ ]:
exact = SansCalculator(6.36e-6).calculate(structure, 'exact average, D2O')
averaged = SansCalculator(6.36e-6, average_amplitude=True).calculate(
    structure, 'averaged amplitude, D2O'
)
for q_value in (0.02, 0.04, 0.06, 0.095, 0.145):
    index = np.abs(exact.q - q_value).argmin()
    ratio = averaged.intensity[index] / exact.intensity[index]
    print(f'q = {exact.q[index]:.3f} Å⁻¹: averaged amplitude / exact = {ratio:.3f}')
ax = exact.plot()
averaged.plot(ax);

## 3. Polydispersity against sasmodels

When SasView fits a measured curve, a spread of sizes comes from
sasmodels: here the `sphere` model with a Gaussian spread of its
radius, `radius_pd = 0.1`, 35 points over ±3σ. coords2sans scales
the whole structure instead, so a sphere of evenly spread carbon
atoms with the same spread must give the same curve.

The atoms fill the sphere evenly: radii R u^(1/3) on a regular u
grid, and each direction twice (+ and −), so the bounding box's
middle is the sphere's centre. The radius is set so that the atoms'
volumes add up to the sphere's.

This check uses the averaged amplitude, which is exact for a sphere
and sees only the radii, so evenly spread radii make a smooth
sphere. The exact average would also count each atom's pair with
itself, which a smooth sphere does not have, and would take about
30 s for 40 000 atoms. Both averages combine the sizes by the same
rule.

In [ ]:
count = 20000
carbon = Structure([1], ['X'], ['C'], [[0.0, 0, 0]], ['C'])
sphere_volume = 2 * count * carbon.volume[0]
radius = (3 * sphere_volume / (4 * np.pi)) ** (1 / 3)
directions = np.random.default_rng(1).normal(size=(count, 3))
directions /= np.linalg.norm(directions, axis=1)[:, None]
radii = radius * ((np.arange(count) + 0.5) / count) ** (1 / 3)
positions = np.r_[radii[:, None] * directions, -radii[:, None] * directions]
carbons = ['C'] * 2 * count
sphere = Structure([1] * 2 * count, ['X'] * 2 * count, carbons, positions, carbons)

carbon_sld = carbon.sld()[0]
solvent_sld = carbon_sld - 1e-6  # a contrast of 1e-6 Å⁻²
calculator = SansCalculator(
    solvent_sld,
    q_min=0.001,
    q_max=0.2,
    q_points=128,
    polydispersity=0.1,
    size_points=35,
    average_amplitude=True,
)
curve = calculator.calculate(sphere, 'coords2sans, sphere of atoms')

sphere_model = DirectModel(empty_data1D(calculator.q), load_model('sphere'))
reference = sphere_model(
    scale=1,
    background=0,
    sld=carbon_sld * 1e6,  # sasmodels counts SLDs in 1e-6 Å⁻²
    sld_solvent=solvent_sld * 1e6,
    radius=radius,
    radius_pd=0.1,
    radius_pd_n=35,
    radius_pd_nsigma=3,
    radius_pd_type='gaussian',
)
print(f'largest relative difference {np.abs(curve.intensity / reference - 1).max():.1e}')
ax = curve.plot()
ax.loglog(calculator.q, reference, 'k:', label='sasmodels sphere')
ax.legend();

## 4. The polyJD recipe

The polyJD script adds 37 curves: one size with weight 0.5, then 36
sizes from a Gaussian of width 0.4, each with its own weight, as
plain added intensities. Its loops are copied below as they are, on
SasView (`NormalDist.inv_cdf` stands in for SciPy's `norm.ppf`; both
give the same sizes).

Three things differ from the sasmodels rule that coords2sans uses:

- The 36 sizes sit at equal steps of the Gaussian's cumulative
  curve, so they already follow the Gaussian of width 0.4. Each size
  is then weighted again, by its step, which favours the middle
  sizes: the spread of the weighted sizes is 0.25, not 0.4.
- The weights add up to 9.8, not 1, so the curve sits high: it is a
  sum, not an average over the sample.
- The smallest size is below zero (−0.10).

In [ ]:
# the script's inputs: q, the PDB data and the solvent SLD; the
# script runs GenSAS with is_avg = True, so coords2sans does too
calculator = SansCalculator(-0.5e-6, average_amplitude=True)
q = calculator.q
pdb_data = sas_gen.PDBReader().read(str(pdb_path))

# SasCalc_polyJD.py from here on, as it is
i = 0.003
m = 0.5142

pos_xMASTER = pdb_data.pos_x.copy()
pos_yMASTER = pdb_data.pos_y.copy()
pos_zMASTER = pdb_data.pos_z.copy()
vol_MASTER = pdb_data.vol_pix.copy()

sas_model = sas_gen.GenSAS()
sas_model.params['solvent_SLD'] = -0.5e-6
sas_model.params['scale'] = 0.5
sas_model.set_sld_data(pdb_data)

sas_model.is_avg = True
averaged_intensity = sas_model.run([q, []])
poly_averaged_intensity = averaged_intensity

for _ in range(0, 18):
    r = NormalDist(1, 0.4).inv_cdf(i)
    pdb_data.pos_x = pos_xMASTER * r
    pdb_data.pos_y = pos_yMASTER * r
    pdb_data.pos_z = pos_zMASTER * r
    pdb_data.vol_pix = vol_MASTER * (r * r * r)
    sas_model.params['scale'] = i
    sas_model.set_sld_data(pdb_data)
    averaged_intensity = sas_model.runXY([q, []])
    poly_averaged_intensity = averaged_intensity + poly_averaged_intensity
    i = i + 0.0284

for _ in range(0, 18):
    r = NormalDist(1, 0.4).inv_cdf(m)
    pdb_data.pos_x = pos_xMASTER * r
    pdb_data.pos_y = pos_yMASTER * r
    pdb_data.pos_z = pos_zMASTER * r
    pdb_data.vol_pix = vol_MASTER * (r * r * r)
    sas_model.params['scale'] = i
    sas_model.set_sld_data(pdb_data)
    averaged_intensity = sas_model.runXY([q, []])
    poly_averaged_intensity = averaged_intensity + poly_averaged_intensity
    m = m + 0.0284
    i = i - 0.0284

In [ ]:
# the script's 37 sizes and weights, as its loops make them
levels = np.r_[0.003 + 0.0284 * np.arange(18), 0.5142 + 0.0284 * np.arange(18)]
sizes = np.r_[1.0, [NormalDist(1, 0.4).inv_cdf(level) for level in levels]]
weights = np.r_[0.5, 0.003 + 0.0284 * np.arange(18), 0.003 + 0.0284 * np.arange(18, 0, -1)]
shares = weights / weights.sum()
spread = np.sqrt(shares @ (sizes - shares @ sizes) ** 2)
print(f'spread of the weighted sizes {spread:.3f}, sum of the weights {weights.sum():.2f},')
print(f'smallest size {sizes.min():.2f}')

one_size = calculator.calculate(structure, 'coords2sans, one size')
calculator.polydispersity = spread
same_spread = calculator.calculate(structure, f'coords2sans, polydispersity {spread:.2f}')
polyjd = poly_averaged_intensity
print(f'at the smallest q, polyJD is {polyjd[0] / one_size.intensity[0]:.1f}× one size,')
print(f'coords2sans with the same spread {same_spread.intensity[0] / one_size.intensity[0]:.1f}×')

ax = one_size.plot()
same_spread.plot(ax)
ax.loglog(q, polyjd, 'k:', label='polyJD script on SasView')
ax.legend();

To compare a curve with a measurement, set `scale` to the sample's
volume fraction of micelles and `polydispersity` to its spread of
sizes, as SasView fits do.